# 8 — CNN BiLSTM
Same N/S/V/F benchmark. Hyperparameters/checkpoints use Validation; all four final models are compared on Test in notebook 9.


## Configuration and helpers


In [ ]:
from pathlib import Path
import json, hashlib, random, time, copy, platform, importlib.metadata, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

VERSION_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'benchmark_config.json').is_file())
CONFIG = json.loads((VERSION_ROOT / 'benchmark_config.json').read_text(encoding='utf-8'))
RAW = Path(CONFIG['raw_directory'])
if not RAW.is_absolute():
    RAW = (VERSION_ROOT / RAW).resolve()
DATA = VERSION_ROOT / 'dataset/processed'
ARTIFACTS = VERSION_ROOT / 'model/artifacts'
REPORTS = VERSION_ROOT / 'model/reports'
CLASSES = CONFIG['classes']
N_CLASSES = len(CLASSES)
SEED = CONFIG['seed']
FS = 360
LEFT, RIGHT = CONFIG['left_samples'], CONFIG['right_samples']
WAVE_LENGTH = LEFT + RIGHT + 1
ALL_SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
ALL_LABEL_MAP = {s: i for i, symbols in enumerate(ALL_SYMBOLS) for s in symbols}
if CLASSES != ['N', 'S', 'V', 'F'] or LEFT < 18 or RIGHT < 180 or not CONFIG['offline_context']:
    raise ValueError('This version requires N/S/V/F, at least 50 ms before / 500 ms after FP, and offline context')

def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; รัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def macro_f1(y, predicted):
    return float(f1_score(y, predicted, labels=list(range(N_CLASSES)), average='macro', zero_division=0))

def class_weights(y, power=0.5, cap=20.0):
    counts = np.bincount(np.asarray(y, dtype=int), minlength=N_CLASSES)
    if (counts == 0).any():
        raise ValueError('Training data must contain all configured classes')
    values = np.minimum((len(y) / (N_CLASSES * counts)) ** power, cap)
    return values / np.average(values, weights=counts)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass

def environment():
    names = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'wfdb', 'pyarrow',
             'PyWavelets', 'xgboost', 'torch', 'optuna']
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = 'not installed'
    return {'python': platform.python_version(), 'platform': platform.platform(), 'packages': versions}

def validate_partitions(frame):
    if set(frame.split) != {'train', 'validation', 'test'}:
        raise ValueError('Require train, validation and test')
    if frame.beat_id.duplicated().any() or frame.groupby('patient_group').split.nunique().max() != 1:
        raise ValueError('Duplicate beat or patient leakage')
    for name, part in frame.groupby('split'):
        if set(part.class_id) != set(range(N_CLASSES)):
            raise ValueError(f'{name}: missing class')
    paired = frame.loc[frame.record_id.isin(['201', '202'])]
    if paired.patient_group.nunique() != 1 or paired.split.nunique() != 1:
        raise ValueError('201 and 202 must remain in one patient group and one partition')

def load_partition(split, *, allow_test=False):
    if split == 'test' and not allow_test:
        raise PermissionError('Test is evaluated only in notebook 9')
    manifest = read_json(DATA / 'prepared.json')
    if manifest['config_hash'] != sha256(VERSION_ROOT / 'benchmark_config.json'):
        raise ValueError('Configuration changed; rerun notebooks 1-4')
    for upstream in ['audit.json', 'split.json', 'preprocessing.json', 'feature_schema.json',
                     'inventory.parquet', 'split_inventory.parquet']:
        if sha256(DATA / upstream) != manifest['files'][upstream]:
            raise ValueError(f'Upstream artifact changed: {upstream}; rerun preparation')
    folder = DATA / split
    values = {}
    for key, name in [('meta', 'metadata.parquet'), ('wave', 'waveforms.npy'),
                      ('rr', 'rr.npy'), ('x', 'features_raw.npy')]:
        if sha256(folder / name) != manifest['files'][f'{split}/{name}']:
            raise ValueError(f'Artifact changed: {split}/{name}; rerun preparation')
        values[key] = pd.read_parquet(folder / name) if key == 'meta' else np.load(folder / name)
    m = values['meta']
    if not all(len(v) == len(m) for v in values.values()):
        raise ValueError('Array lengths differ')
    if not np.array_equal(m.waveform_index, np.arange(len(m))) or not m.split.eq(split).all():
        raise ValueError('Array/metadata alignment failed')
    if values['wave'].shape != (len(m), 2, WAVE_LENGTH) or not np.isfinite(values['wave']).all():
        raise ValueError('Invalid waveform shape or nonfinite waveform')
    values['y'] = m.class_id.to_numpy(dtype=np.int64)
    return values

def concatenate_partitions(parts):
    return {key: pd.concat([p[key] for p in parts], ignore_index=True) if key == 'meta'
            else np.concatenate([p[key] for p in parts]) for key in parts[0]}

def fit_transforms(part):
    feature_imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    rr_imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    rr_scaler = RobustScaler(quantile_range=(10., 90.))
    feature_imputer.fit(part['x'])
    rr_scaler.fit(rr_imputer.fit_transform(part['rr']))
    return {'feature_imputer': feature_imputer, 'rr_imputer': rr_imputer, 'rr_scaler': rr_scaler}

def transform_partition(part, transforms):
    result = dict(part)
    result['x'] = transforms['feature_imputer'].transform(part['x']).astype(np.float32)
    filled = transforms['rr_imputer'].transform(part['rr'])
    scaled = transforms['rr_scaler'].transform(filled)
    # Clipping bounds are fixed protocol constants; missing flags keep edge information.
    result['context'] = np.column_stack([np.clip(scaled, -10., 10.),
                                        ~np.isfinite(part['rr'])]).astype(np.float32)
    if not np.isfinite(result['x']).all() or not np.isfinite(result['context']).all():
        raise ValueError('Imputation/scaling produced nonfinite input')
    return result

def prepare_training(train_raw, validation_raw):
    transforms = fit_transforms(train_raw)
    return transform_partition(train_raw, transforms), transform_partition(validation_raw, transforms), transforms

def metrics(y, pred):
    report = classification_report(y, pred, labels=list(range(N_CLASSES)), target_names=CLASSES,
                                   output_dict=True, zero_division=0)
    matrix = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    normalized = np.divide(matrix, matrix.sum(1, keepdims=True), out=np.zeros_like(matrix, dtype=float),
                           where=matrix.sum(1, keepdims=True) != 0)
    return {'accuracy': float(accuracy_score(y, pred)), 'macro_f1': macro_f1(y, pred),
            'weighted_f1': float(report['weighted avg']['f1-score']),
            'balanced_accuracy': float(normalized.diagonal().mean()), 'per_class': report,
            'confusion_matrix': matrix.tolist(), 'normalized_confusion_matrix': normalized.tolist()}

def plot_trials(study, destination):
    frame = study.trials_dataframe()
    frame.to_csv(destination / 'trials.csv', index=False)
    complete = frame.loc[frame.state.eq('COMPLETE')].copy()
    if len(complete):
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(complete.number, complete.value, 'o-', alpha=.6, label='Validation Macro-F1')
        ax.plot(complete.number, complete.value.cummax(), linewidth=2, label='Best so far')
        ax.set(xlabel='Trial', ylabel='Macro-F1', ylim=(0, 1), title=study.study_name)
        ax.legend(); ax.grid(alpha=.2); fig.tight_layout()
        fig.savefig(destination / 'tuning.png', dpi=160)
        plt.show(); plt.close(fig)


In [ ]:
MODEL='cnn_bilstm'
MODEL_REVISION=2


## Baseline hyperparameters


In [ ]:
BASELINE_PARAMETERS={'base_channels': 32, 'stages': 3, 'blocks_per_stage': 1, 'kernel': 7, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128, 'rr_hidden': 32, 'head_hidden': 128, 'class_weight_power': 0.5, 'label_smoothing': 0.0, 'lstm_hidden': 64, 'lstm_layers': 1}


## Optuna search space


In [ ]:
SEARCH_SPACE={'base_channels': ['categorical', [16, 32, 64]], 'stages': ['categorical', [2, 3]], 'blocks_per_stage': ['categorical', [1, 2]], 'kernel': ['categorical', [5, 7, 11, 15]], 'dropout': ['float', 0.1, 0.5], 'lr': ['log', 0.0001, 0.003], 'weight_decay': ['log', 1e-06, 0.01], 'batch_size': ['categorical', [64, 128, 256]], 'rr_hidden': ['categorical', [16, 32, 64]], 'head_hidden': ['categorical', [64, 128, 256]], 'class_weight_power': ['categorical', [0.0, 0.5, 1.0]], 'label_smoothing': ['categorical', [0.0, 0.02]], 'lstm_hidden': ['categorical', [32, 64, 128]], 'lstm_layers': ['categorical', [1, 2]]}


## Training controls


In [ ]:
import torch
DL_DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
N_TRIALS=80
MAX_EPOCHS=200
MIN_EPOCHS=20
EARLY_STOPPING_PATIENCE=25
PRUNING_WARMUP=20
USE_AMP=True
AUGMENTATION={'amplitude':.1,'shift_samples':3,'noise_std':.01}
REFIT_ON_DEVELOPMENT=True
TRAINING_SIGNATURE={'max_epochs':MAX_EPOCHS,'min_epochs':MIN_EPOCHS,'patience':EARLY_STOPPING_PATIENCE,
                    'augmentation':AUGMENTATION,'amp':USE_AMP,'seed':SEED}
print('DL device:',DL_DEVICE)


## Model and training definitions


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel, stride=stride, padding=kernel//2, bias=False),
            nn.BatchNorm1d(out_channels), nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, kernel, padding=kernel//2, bias=False),
            nn.BatchNorm1d(out_channels))
        self.skip = nn.Identity() if in_channels == out_channels and stride == 1 else nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False), nn.BatchNorm1d(out_channels))
    def forward(self, x):
        return F.relu(self.main(x) + self.skip(x))

class ECGNetwork(nn.Module):
    def __init__(self, kind, parameters, context_dim, n_classes=N_CLASSES):
        super().__init__()
        layers, previous = [], 2
        for stage in range(parameters['stages']):
            width = min(parameters['base_channels'] * 2**stage, 256)
            for block in range(parameters['blocks_per_stage']):
                layers.append(ResidualBlock(previous, width, parameters['kernel'], stride=2 if block == 0 else 1))
                previous = width
        self.conv = nn.Sequential(*layers)
        self.kind = kind
        if kind == 'cnn_bilstm':
            hidden = parameters['lstm_hidden']
            self.lstm = nn.LSTM(previous, hidden, num_layers=parameters['lstm_layers'],
                batch_first=True, bidirectional=True,
                dropout=parameters['dropout'] if parameters['lstm_layers'] > 1 else 0.)
            previous = 2 * hidden
        self.rr_branch = nn.Sequential(nn.Linear(context_dim, parameters['rr_hidden']),
                                      nn.LayerNorm(parameters['rr_hidden']), nn.ReLU())
        combined = 2 * previous + parameters['rr_hidden']
        self.head = nn.Sequential(nn.Linear(combined, parameters['head_hidden']),
            nn.LayerNorm(parameters['head_hidden']), nn.ReLU(), nn.Dropout(parameters['dropout']),
            nn.Linear(parameters['head_hidden'], n_classes))
    def forward(self, waveform, context):
        encoded = self.conv(waveform)
        if self.kind == 'cnn_bilstm':
            encoded, _ = self.lstm(encoded.transpose(1, 2))
            encoded = encoded.transpose(1, 2)
        pooled = torch.cat([encoded.mean(2), encoded.amax(2)], dim=1)
        return self.head(torch.cat([pooled, self.rr_branch(context)], dim=1))

def augment(waveforms):
    if AUGMENTATION['amplitude']:
        # Record-normalized channel keeps relative beat amplitude; channel 1 stays shape-normalized.
        waveforms = waveforms.clone()
        amount = AUGMENTATION['amplitude']
        scale = 1 + (torch.rand(len(waveforms), 1, device=waveforms.device)*2-1)*amount
        waveforms[:, 0] *= scale
    if AUGMENTATION['shift_samples']:
        shift = AUGMENTATION['shift_samples']
        padded = F.pad(waveforms, (shift, shift), mode='reflect')
        offsets = torch.randint(-shift, shift+1, (len(waveforms), 1, 1), device=waveforms.device)
        index = torch.arange(waveforms.shape[-1], device=waveforms.device)[None,None,:] + shift + offsets
        waveforms = padded.gather(2, index.expand(-1, waveforms.shape[1], -1))
    if AUGMENTATION['noise_std']:
        waveforms = waveforms + torch.randn_like(waveforms)*AUGMENTATION['noise_std']
    return waveforms

def predict_probabilities(model, part, batch_size=256):
    model.eval()
    device = next(model.parameters()).device
    result = []
    with torch.inference_mode():
        for start in range(0, len(part['y']), batch_size):
            wave = torch.as_tensor(part['wave'][start:start+batch_size], device=device)
            context = torch.as_tensor(part['context'][start:start+batch_size], device=device)
            result.append(model(wave, context).softmax(1).cpu().numpy())
    return np.concatenate(result)

def fit_model(parameters, train, validation=None, *, trial=None, fixed_steps=None):
    seed_everything()
    device = torch.device(DL_DEVICE)
    model = ECGNetwork(MODEL, parameters, train['context'].shape[1]).to(device)
    weights = class_weights(train['y'], parameters['class_weight_power'])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device),
                                    label_smoothing=parameters['label_smoothing'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=parameters['lr'], weight_decay=parameters['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=parameters['lr']*.01)
    amp = USE_AMP and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    dataset = TensorDataset(torch.from_numpy(train['wave']), torch.from_numpy(train['context']),
                            torch.from_numpy(train['y']))
    loader = DataLoader(dataset, batch_size=parameters['batch_size'], shuffle=True, num_workers=0,
                        pin_memory=device.type == 'cuda', generator=torch.Generator().manual_seed(SEED))
    best_score, stale, best_steps, best_state = -1., 0, 0, None
    history = []
    for epoch in range(1, (fixed_steps or MAX_EPOCHS)+1):
        model.train()
        loss_sum, loss_denominator = 0., 0.
        for wave, context, y in loader:
            wave, context, y = wave.to(device), context.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model(augment(wave), context)
                loss = criterion(logits.float(), y)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            denominator = float(criterion.weight[y].sum().item())
            loss_sum += float(loss.item())*denominator
            loss_denominator += denominator
        row = {'epoch': epoch, 'train_loss': loss_sum/loss_denominator, 'lr': optimizer.param_groups[0]['lr']}
        if validation is not None:
            probabilities = predict_probabilities(model, validation)
            score = macro_f1(validation['y'], probabilities.argmax(1))
            row['validation_macro_f1'] = score
            if score > best_score + 1e-6:
                best_score, stale, best_steps = score, 0, epoch
                best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            else:
                stale += 1
            if trial is not None:
                trial.report(score, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned(f'DL pruned at epoch {epoch}')
        history.append(row)
        scheduler.step()
        if validation is not None and epoch >= MIN_EPOCHS and stale >= EARLY_STOPPING_PATIENCE:
            break
    if validation is not None:
        model.load_state_dict(best_state)
    else:
        best_steps = fixed_steps
    return model, {'best_steps': int(best_steps), 'class_weights': weights.tolist(), 'device': str(device)}, history

def save_model(model, destination):
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()}, destination / 'model.pt')

def load_model(destination, parameters, context_dim):
    model = ECGNetwork(MODEL, parameters, context_dim)
    model.load_state_dict(torch.load(destination / 'model.pt', map_location='cpu', weights_only=True))
    return model.to(DL_DEVICE).eval()


## Optuna and final refit definitions


In [ ]:
import optuna
import gc

def suggest_parameters(trial):
    params = {}
    for name, specification in SEARCH_SPACE.items():
        kind = specification[0]
        if kind == 'categorical':
            params[name] = trial.suggest_categorical(name, specification[1])
        elif kind == 'int':
            params[name] = trial.suggest_int(name, specification[1], specification[2],
                                           step=specification[3] if len(specification)>3 else 1)
        else:
            params[name] = trial.suggest_float(name, specification[1], specification[2],
                                               log=kind == 'log')
    return params

def run_tuning(train, validation, transforms):
    prepared_hash = sha256(DATA / 'prepared.json')
    signature = hashlib.sha256(json.dumps({'prepared': prepared_hash, 'model': MODEL,
        'space': SEARCH_SPACE, 'revision': MODEL_REVISION, 'baseline': BASELINE_PARAMETERS,
        'controls': TRAINING_SIGNATURE}, sort_keys=True).encode()).hexdigest()
    run_id = signature[:16]
    destination = ARTIFACTS / MODEL / run_id
    destination.mkdir(parents=True, exist_ok=True)
    best_folder = destination / 'validation_best'
    best_folder.mkdir(exist_ok=True)
    storage = 'sqlite:///' + (destination / 'study.sqlite3').as_posix()
    study = optuna.create_study(study_name=f'{MODEL}_{run_id}', direction='maximize', storage=storage,
        load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=15,
            multivariate=True, group=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=15,
            n_warmup_steps=PRUNING_WARMUP, interval_steps=5))
    if study.user_attrs.get('signature', signature) != signature:
        raise ValueError('Study protocol/data mismatch')
    study.set_user_attr('signature', signature)
    if not study.trials:
        study.enqueue_trial(BASELINE_PARAMETERS)
    def objective(trial):
        params = suggest_parameters(trial)
        seed_everything()
        started = time.perf_counter()
        model = None
        try:
            model, info, history = fit_model(params, train, validation, trial=trial)
            probabilities = predict_probabilities(model, validation)
            score = macro_f1(validation['y'], probabilities.argmax(1))
            info.update({'model': MODEL, 'parameters': params, 'validation_macro_f1': score,
                'trial': trial.number, 'train_seconds': time.perf_counter()-started,
                'prepared_hash': prepared_hash, 'signature': signature, 'classes': CLASSES,
                'context_dim': train['context'].shape[1], 'seed': SEED, 'environment': environment()})
            trial.set_user_attr('best_steps', info['best_steps'])
            trial.set_user_attr('train_seconds', info['train_seconds'])
            write_json(destination / f'history_{trial.number:04d}.json', history)
            previous = read_json(best_folder/'info.json') if (best_folder/'info.json').exists() else None
            if previous is None or score > previous['validation_macro_f1']:
                save_model(model, best_folder)
                joblib.dump(transforms, best_folder / 'transforms.joblib')
                write_json(best_folder/'info.json', info)
                frame = validation['meta'].copy()
                frame['predicted'] = probabilities.argmax(1)
                for i, name in enumerate(CLASSES): frame[f'score_{name}'] = probabilities[:,i]
                frame.to_parquet(best_folder/'validation_predictions.parquet', index=False)
            return score
        except RuntimeError as error:
            if 'out of memory' in str(error).lower():
                raise optuna.TrialPruned(f'Memory limit: {error}') from error
            raise
        finally:
            del model
            gc.collect()
            if MODEL in ['cnn','cnn_bilstm'] and torch.cuda.is_available(): torch.cuda.empty_cache()
    finished = sum(t.state.is_finished() for t in study.trials)
    remaining = max(0, N_TRIALS-finished)
    if remaining:
        study.optimize(objective, n_trials=remaining, n_jobs=1, gc_after_trial=True)
    if not (best_folder/'info.json').exists():
        raise RuntimeError('No completed trial')
    info = read_json(best_folder/'info.json')
    if not np.isclose(info['validation_macro_f1'], study.best_value):
        raise ValueError('Saved best checkpoint does not match Optuna study')
    plot_trials(study, destination)
    write_json(ARTIFACTS / MODEL / 'selected.json', {'run_id':run_id, 'path':str(destination.relative_to(ARTIFACTS)),
        'validation_macro_f1':info['validation_macro_f1'], 'parameters':info['parameters'],
        'prepared_hash':prepared_hash, 'signature':signature})
    return study, destination, info

def finalize_model(train_raw, validation_raw, destination, selected_info):
    final = destination / 'final'
    expected = {'signature':selected_info['signature'], 'refit_on_development':REFIT_ON_DEVELOPMENT,
                'selected_trial':selected_info['trial'], 'parameters':selected_info['parameters']}
    model_name = 'model.joblib' if MODEL == 'rf' else 'model.ubj' if MODEL == 'xgb' else 'model.pt'
    if all((final/name).exists() for name in ['info.json', 'transforms.joblib', model_name]):
        existing = read_json(final/'info.json')
        if all(existing.get(k)==v for k,v in expected.items()):
            print('Final model already saved:', final)
            return existing
    final.mkdir(exist_ok=True)
    if REFIT_ON_DEVELOPMENT:
        # Fixed hyperparameters and fixed epoch/round count; no Test is read during refit.
        development_raw = concatenate_partitions([train_raw, validation_raw])
        transforms = fit_transforms(development_raw)
        development = transform_partition(development_raw, transforms)
        seed_everything()
        model, fit_info, history = fit_model(selected_info['parameters'], development,
                                            fixed_steps=selected_info['best_steps'])
        save_model(model, final)
        info = {**selected_info, **fit_info, 'fit_partition':'train+validation', 'training_beats':len(development['y'])}
        write_json(final/'history.json', history)
    else:
        transforms = joblib.load(destination/'validation_best/transforms.joblib')
        model = load_model(destination/'validation_best', selected_info['parameters'], selected_info['context_dim'])
        save_model(model, final)
        info = {**selected_info, 'fit_partition':'train', 'training_beats':len(train_raw['y'])}
    joblib.dump(transforms, final/'transforms.joblib')
    info.update(expected)
    write_json(final/'info.json', info)
    del model
    gc.collect()
    if MODEL in ['cnn','cnn_bilstm'] and torch.cuda.is_available(): torch.cuda.empty_cache()
    return info


## Load Train and Validation


In [ ]:
train_raw=load_partition('train')
validation_raw=load_partition('validation')
train,validation,transforms=prepare_training(train_raw,validation_raw)
print('Train:',len(train['y']),'Validation:',len(validation['y']))


## Baseline and tuning — resumable study


In [ ]:
# Trial 0 uses the baseline; subsequent trials search the declared space.
# N_TRIALS is the total study budget, including completed/pruned trials and baseline.
study,destination,selected_info=run_tuning(train,validation,transforms)
display(pd.DataFrame([selected_info['parameters']]))
print('Best Validation Macro-F1:',selected_info['validation_macro_f1'])


## Refit selected settings on development


In [ ]:
final_info=finalize_model(train_raw,validation_raw,destination,selected_info)
print('Final fit:',final_info['fit_partition'],'beats:',final_info['training_beats'])
print('Fixed epochs/rounds:',final_info['best_steps'])


## Selected training history


In [ ]:
history=read_json(destination/f"history_{selected_info['trial']:04d}.json")
if isinstance(history,list):
    frame=pd.DataFrame(history)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].plot(frame.epoch,frame.train_loss); axes[0].set(xlabel='Epoch',ylabel='Train loss')
    axes[1].plot(frame.epoch,frame.validation_macro_f1)
    axes[1].axvline(selected_info['best_steps'],linestyle='--',color='black')
    axes[1].set(xlabel='Epoch',ylabel='Validation Macro-F1',ylim=(0,1))
    fig.tight_layout(); fig.savefig(destination/'selected_history.png',dpi=160); plt.show(); plt.close(fig)
elif 'validation' in history:
    frame=pd.DataFrame(history['validation']); frame.plot(subplots=True,figsize=(9,6))
    plt.tight_layout(); plt.savefig(destination/'selected_history.png',dpi=160); plt.show(); plt.close('all')
else:
    display(pd.DataFrame([history]))
